### The testing Raw FHIR Bundles (JSON file of patient bundles) can be syntheted by $ git clone https://github.com/synthetichealth/synthea.git, then $ ./run_synthea -p 20  #-p (patient size)).

## This pipeline ingested FHIR bundles, extracted multiple clinical resource types, normalized them into a flat event table, built both full and cleaned longitudinal timelines, scaled from 1 patient to all patients, save the outputs and create a multi-patient-level summary table.

In [1]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd



In [2]:
# check current dir path
Path.cwd()

PosixPath('/home/jiawu2/projects/clinical-fhir-timeline')

In [3]:
#set corect row data dir
fhir_dir = Path("data/synthea_raw/fhir")
files = sorted(fhir_dir.glob("*.json"))   
len(files), [f.name for f in files[:5]]

(24,
 ['Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd.json',
  'Bobby524_Nikolaus26_e94ea1e4-7725-20f8-c494-32a002550372.json',
  'Carry843_Jerrie417_Bosco882_8f919ef5-4994-e505-786f-fd2d5a4d895f.json',
  'Cornell131_Ratke343_57ee3fbe-8320-2480-7b90-0e0980bd584c.json',
  'Garfield38_Ferry570_83a1f980-5e41-7be5-445a-882e773a70dc.json'])

In [4]:
# inspect one patient bundle, understand its structure
example_file = files[0]
example_file

PosixPath('data/synthea_raw/fhir/Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd.json')

In [5]:
# load the one patient's JSON into Python
with open(example_file) as fh:
    data = json.load(fh)   #JSON text into Python object

type(data), list(data.keys())[:20]

(dict, ['resourceType', 'type', 'entry'])

In [6]:
data.get("resourceType"), len(data.get("entry", []))

('Bundle', 1009)

In [7]:
#count the resource types inside the bundle 
counter = Counter()

for entry in data.get("entry", []):
    rtype = entry.get("resource", {}).get("resourceType", "UNKNOWN")   #gets the embedded FHIR resouces and type as dict
    counter[rtype] += 1

counter

Counter({'Observation': 402,
         'Procedure': 133,
         'DiagnosticReport': 105,
         'Claim': 73,
         'ExplanationOfBenefit': 73,
         'Encounter': 46,
         'DocumentReference': 46,
         'Condition': 42,
         'MedicationRequest': 27,
         'SupplyDelivery': 16,
         'Immunization': 12,
         'Device': 9,
         'ImagingStudy': 5,
         'Medication': 5,
         'MedicationAdministration': 5,
         'CareTeam': 4,
         'CarePlan': 4,
         'Patient': 1,
         'Provenance': 1})

In [8]:
# preview first few resources
for i, entry in enumerate(data.get("entry", [])[:5]):
    r = entry.get("resource", {})
    print(i, r.get("resourceType"), list(r.keys())[:12])

0 Patient ['resourceType', 'id', 'meta', 'text', 'extension', 'identifier', 'name', 'telecom', 'gender', 'birthDate', 'address', 'maritalStatus']
1 Encounter ['resourceType', 'id', 'meta', 'identifier', 'status', 'class', 'type', 'subject', 'participant', 'period', 'location', 'serviceProvider']
2 Condition ['resourceType', 'id', 'meta', 'clinicalStatus', 'verificationStatus', 'category', 'code', 'subject', 'encounter', 'onsetDateTime', 'recordedDate']
3 DiagnosticReport ['resourceType', 'id', 'meta', 'status', 'category', 'code', 'subject', 'encounter', 'effectiveDateTime', 'issued', 'performer', 'presentedForm']
4 DocumentReference ['resourceType', 'id', 'meta', 'identifier', 'status', 'type', 'category', 'subject', 'date', 'author', 'custodian', 'content']


In [9]:
def get_code_text(resource):
    code = resource.get("code", {})
    if "text" in code:
        return code["text"]
    coding = code.get("coding", [])
    if coding:
        return coding[0].get("display") or coding[0].get("code")
    return None

def get_code_value(resource):
    code = resource.get("code", {})
    coding = code.get("coding", [])
    if coding:
        return coding[0].get("code")
    return None

def get_encounter_ref(resource):
    return resource.get("encounter", {}).get("reference")

def get_subject_ref(resource):
    return resource.get("subject", {}).get("reference")

def get_patient_name(patient_resource):
    names = patient_resource.get("name", [])
    if not names:
        return None
    n = names[0]
    given = " ".join(n.get("given", []))
    family = n.get("family", "")
    return f"{given} {family}".strip()

def normalize_date(value):
    if value is None:
        return None
    return str(value)

def get_vaccine_text(resource):
    vaccine = resource.get("vaccineCode", {})
    if "text" in vaccine:
        return vaccine["text"]
    coding = vaccine.get("coding", [])
    if coding:
        return coding[0].get("display") or coding[0].get("code")
    return None

In [10]:
#inspect and separate resources
entries = data.get("entry", [])
resources = [e.get("resource", {}) for e in entries]

patient_resources = [r for r in resources if r.get("resourceType") == "Patient"]
encounters = [r for r in resources if r.get("resourceType") == "Encounter"]
conditions = [r for r in resources if r.get("resourceType") == "Condition"]
observations = [r for r in resources if r.get("resourceType") == "Observation"]
med_requests = [r for r in resources if r.get("resourceType") == "MedicationRequest"]
procedures = [r for r in resources if r.get("resourceType") == "Procedure"]
immunizations = [r for r in resources if r.get("resourceType") == "Immunization"]

len(patient_resources), len(encounters), len(conditions), len(observations), len(med_requests), len(procedures), len(immunizations)

(1, 46, 42, 402, 27, 133, 12)

In [11]:
#basic patient info
patient = patient_resources[0]
patient_id = patient.get("id")
patient_name = get_patient_name(patient)

patient_id, patient_name

('55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd', 'Ardelle563 Lauri399 Aufderhar910')

In [12]:
#improve Encounter labels (previous labels are NaN). 
def get_encounter_label(resource):
    types = resource.get("type", [])
    if types:
        first = types[0]
        text = first.get("text")
        if text:
            return text
        coding = first.get("coding", [])
        if coding:
            return coding[0].get("display") or coding[0].get("code")
    
    enc_class = resource.get("class", {})
    if isinstance(enc_class, dict):
        return enc_class.get("display") or enc_class.get("code") or "Encounter"
    
    return "Encounter"


In [13]:
# build encouter rows- Creates a list of flat dictionaries, one per encounter.This is the first step of converting nested FHIR into a tidy table.
encounter_rows = []

for r in encounters:
    row = {
        "patient_id": patient_id,
        "patient_name": patient_name,
        "resource_type": "Encounter",
        "resource_id": r.get("id"),
        "date": normalize_date(r.get("period", {}).get("start")),
        "label": get_encounter_label(r),
        "code": get_code_value(r),
        "encounter_ref": f"Encounter/{r.get('id')}" if r.get("id") else None,
        "subject_ref": get_subject_ref(r),
    }
    encounter_rows.append(row)

pd.DataFrame(encounter_rows).head()

,patient_id,patient_name,resource_type,resource_id,date,label,code,encounter_ref,subject_ref
0,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Encounter,55d4e20e-10c7-1d0f-7533-e42d4eae3c5d,1965-12-17T16:33:40-08:00,General examination of patient (procedure),None,Encounter/55d4e20e-10c7-1d0f-7533-e42d4eae3c5d,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
1,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Encounter,55d4e20e-10c7-1d0f-d25d-1f073d4ed1eb,1969-04-11T16:33:40-08:00,Prenatal visit (regime/therapy),None,Encounter/55d4e20e-10c7-1d0f-d25d-1f073d4ed1eb,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
2,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Encounter,55d4e20e-10c7-1d0f-cb8d-9b74f706fbe4,1987-10-30T16:33:40-08:00,General examination of patient (procedure),None,Encounter/55d4e20e-10c7-1d0f-cb8d-9b74f706fbe4,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
3,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Encounter,55d4e20e-10c7-1d0f-d9dc-2aae189ec7f2,2003-10-20T02:33:40-07:00,Encounter for symptom (procedure),None,Encounter/55d4e20e-10c7-1d0f-d9dc-2aae189ec7f2,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
4,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Encounter,55d4e20e-10c7-1d0f-f8aa-93427b44527b,2013-02-08T16:33:40-08:00,Encounter for problem (procedure),None,Encounter/55d4e20e-10c7-1d0f-f8aa-93427b44527b,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd


In [14]:
#build conditon rows - timeline rows for diagnoses/problems.
condition_rows = []

for r in conditions:
    row = {
        "patient_id": patient_id,
        "patient_name": patient_name,
        "resource_type": "Condition",
        "resource_id": r.get("id"),
        "date": normalize_date(r.get("onsetDateTime") or r.get("recordedDate")),
        "label": get_code_text(r),
        "code": get_code_value(r),
        "encounter_ref": get_encounter_ref(r),
        "subject_ref": get_subject_ref(r),
    }
    condition_rows.append(row)

pd.DataFrame(condition_rows).head()

,patient_id,patient_name,resource_type,resource_id,date,label,code,encounter_ref,subject_ref
0,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Condition,55d4e20e-10c7-1d0f-90c9-5eab66e6fb18,1965-12-17T17:04:41-08:00,Received higher education (finding),224299000,urn:uuid:55d4e20e-10c7-1d0f-7533-e42d4eae3c5d,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
1,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Condition,55d4e20e-10c7-1d0f-ff69-8355babc28f5,1969-04-23T20:58:04-08:00,Recurrent urinary tract infection (disorder),197927001,urn:uuid:55d4e20e-10c7-1d0f-d25d-1f073d4ed1eb,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
2,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Condition,55d4e20e-10c7-1d0f-aa1a-f81005e1d9c3,1987-10-30T17:16:05-08:00,Refugee (person),446654005,urn:uuid:55d4e20e-10c7-1d0f-cb8d-9b74f706fbe4,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
3,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Condition,55d4e20e-10c7-1d0f-ce82-dda9d67fc77d,2003-10-20T02:33:40-07:00,Chronic sinusitis (disorder),40055000,urn:uuid:55d4e20e-10c7-1d0f-d9dc-2aae189ec7f2,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
4,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Condition,55d4e20e-10c7-1d0f-b9b8-78e76a079de0,2013-02-08T16:33:40-08:00,Loss of teeth (disorder),37320007,urn:uuid:55d4e20e-10c7-1d0f-f8aa-93427b44527b,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd


In [15]:
#build observation rows - timeline rows for labs/vitals/measurements.
observation_rows = []

for r in observations:
    row = {
        "patient_id": patient_id,
        "patient_name": patient_name,
        "resource_type": "Observation",
        "resource_id": r.get("id"),
        "date": normalize_date(
            r.get("effectiveDateTime")
            or r.get("issued")
            or r.get("effectivePeriod", {}).get("start")
        ),
        "label": get_code_text(r),
        "code": get_code_value(r),
        "encounter_ref": get_encounter_ref(r),
        "subject_ref": get_subject_ref(r),
    }
    observation_rows.append(row)

pd.DataFrame(observation_rows).head()

,patient_id,patient_name,resource_type,resource_id,date,label,code,encounter_ref,subject_ref
0,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Observation,55d4e20e-10c7-1d0f-7e59-7d191ad110a1,2016-11-25T16:33:40-08:00,Cholesterol [Mass/volume] in Serum or Plasma,2093-3,urn:uuid:55d4e20e-10c7-1d0f-87eb-d1cd7ac529ad,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
1,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Observation,55d4e20e-10c7-1d0f-00b0-a3eabc0057f8,2016-11-25T16:33:40-08:00,Triglyceride [Mass/volume] in Serum or Plasma,2571-8,urn:uuid:55d4e20e-10c7-1d0f-87eb-d1cd7ac529ad,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
2,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Observation,55d4e20e-10c7-1d0f-ed93-796f161c2ef2,2016-11-25T16:33:40-08:00,Cholesterol in LDL [Mass/volume] in Serum or P...,18262-6,urn:uuid:55d4e20e-10c7-1d0f-87eb-d1cd7ac529ad,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
3,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Observation,55d4e20e-10c7-1d0f-b356-61e7f99ca873,2016-11-25T16:33:40-08:00,Cholesterol in HDL [Mass/volume] in Serum or P...,2085-9,urn:uuid:55d4e20e-10c7-1d0f-87eb-d1cd7ac529ad,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
4,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Observation,55d4e20e-10c7-1d0f-1f17-a9515c55d730,2016-11-25T16:33:40-08:00,Glucose [Mass/volume] in Blood,2339-0,urn:uuid:55d4e20e-10c7-1d0f-87eb-d1cd7ac529ad,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd


In [16]:
#build medication request rows - timeline rows for medications (slightly different in FHIR).
medication_rows = []

for r in med_requests:
    med_label = None
    
    med_ref = r.get("medicationReference", {})
    med_code = r.get("medicationCodeableConcept", {})
    
    if "text" in med_code:
        med_label = med_code["text"]
    else:
        coding = med_code.get("coding", [])
        if coding:
            med_label = coding[0].get("display") or coding[0].get("code")

    row = {
        "patient_id": patient_id,
        "patient_name": patient_name,
        "resource_type": "MedicationRequest",
        "resource_id": r.get("id"),
        "date": normalize_date(r.get("authoredOn")),
        "label": med_label,
        "code": None,
        "encounter_ref": get_encounter_ref(r),
        "subject_ref": get_subject_ref(r),
    }
    medication_rows.append(row)

pd.DataFrame(medication_rows).head()

,patient_id,patient_name,resource_type,resource_id,date,label,code,encounter_ref,subject_ref
0,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,MedicationRequest,55d4e20e-10c7-1d0f-e339-2fc4d53e0adb,2014-01-31T16:33:40-08:00,Clopidogrel 75 MG Oral Tablet,None,urn:uuid:55d4e20e-10c7-1d0f-fb48-14cecb9c9a96,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
1,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,MedicationRequest,55d4e20e-10c7-1d0f-0fb0-a0c4ee6a692f,2014-01-31T16:33:40-08:00,Simvastatin 20 MG Oral Tablet,None,urn:uuid:55d4e20e-10c7-1d0f-fb48-14cecb9c9a96,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
2,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,MedicationRequest,55d4e20e-10c7-1d0f-d94c-a4c47bf28579,2014-01-31T16:33:40-08:00,24 HR metoprolol succinate 100 MG Extended Rel...,None,urn:uuid:55d4e20e-10c7-1d0f-fb48-14cecb9c9a96,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
3,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,MedicationRequest,55d4e20e-10c7-1d0f-e171-57d650747554,2014-01-31T16:33:40-08:00,Nitroglycerin 0.4 MG/ACTUAT Mucosal Spray,None,urn:uuid:55d4e20e-10c7-1d0f-fb48-14cecb9c9a96,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
4,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,MedicationRequest,55d4e20e-10c7-1d0f-8353-6f7afc5870e8,2016-12-16T16:33:40-08:00,Simvastatin 10 MG Oral Tablet,None,urn:uuid:55d4e20e-10c7-1d0f-7c9c-825006e99b0a,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd


In [17]:
#build procedure rows -timeline rows for procedures/interventions.
procedure_rows = []

for r in procedures:
    performed = r.get("performedDateTime")
    if not performed:
        performed = r.get("performedPeriod", {}).get("start")

    row = {
        "patient_id": patient_id,
        "patient_name": patient_name,
        "resource_type": "Procedure",
        "resource_id": r.get("id"),
        "date": normalize_date(performed),
        "label": get_code_text(r),
        "code": get_code_value(r),
        "encounter_ref": get_encounter_ref(r),
        "subject_ref": get_subject_ref(r),
    }
    procedure_rows.append(row)

pd.DataFrame(procedure_rows).head()

,patient_id,patient_name,resource_type,resource_id,date,label,code,encounter_ref,subject_ref
0,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Procedure,55d4e20e-10c7-1d0f-d2e6-26807f353525,2016-11-25T16:33:40-08:00,Assessment of health and social care needs (pr...,710824005,urn:uuid:55d4e20e-10c7-1d0f-87eb-d1cd7ac529ad,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
1,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Procedure,55d4e20e-10c7-1d0f-5871-80e05cf9d6c0,2016-11-25T17:12:26-08:00,Assessment using Morse Fall Scale (procedure),762993000,urn:uuid:55d4e20e-10c7-1d0f-87eb-d1cd7ac529ad,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
2,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Procedure,55d4e20e-10c7-1d0f-433a-6e0cf3f4b641,2016-11-25T17:31:43-08:00,Depression screening (procedure),171207006,urn:uuid:55d4e20e-10c7-1d0f-87eb-d1cd7ac529ad,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
3,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Procedure,55d4e20e-10c7-1d0f-2521-6008a434c2cf,2016-11-25T17:44:16-08:00,Depression screening (procedure),171207006,urn:uuid:55d4e20e-10c7-1d0f-87eb-d1cd7ac529ad,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
4,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Procedure,55d4e20e-10c7-1d0f-ef22-5f37e41d688c,2016-11-25T18:09:01-08:00,Assessment of substance use (procedure),428211000124100,urn:uuid:55d4e20e-10c7-1d0f-87eb-d1cd7ac529ad,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd


In [18]:
#build immunization rows
immunization_rows = []

for r in immunizations:
    row = {
        "patient_id": patient_id,
        "patient_name": patient_name,
        "resource_type": "Immunization",
        "resource_id": r.get("id"),
        "date": normalize_date(r.get("occurrenceDateTime")),
        "label": get_vaccine_text(r),
        "code": None,
        "encounter_ref": get_encounter_ref(r),
        "subject_ref": get_subject_ref(r),
    }
    immunization_rows.append(row)

pd.DataFrame(immunization_rows).head()

,patient_id,patient_name,resource_type,resource_id,date,label,code,encounter_ref,subject_ref
0,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Immunization,55d4e20e-10c7-1d0f-0c28-387de8676deb,2016-11-25T16:33:40-08:00,"Influenza, split virus, trivalent, PF",None,urn:uuid:55d4e20e-10c7-1d0f-87eb-d1cd7ac529ad,None
1,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Immunization,55d4e20e-10c7-1d0f-7dfc-b14a9be20301,2018-02-16T16:33:40-08:00,"Influenza, split virus, trivalent, PF",None,urn:uuid:55d4e20e-10c7-1d0f-141a-e938013ae1ec,None
2,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Immunization,55d4e20e-10c7-1d0f-014b-c47504655fa5,2019-02-22T16:33:40-08:00,"Influenza, split virus, trivalent, PF",None,urn:uuid:55d4e20e-10c7-1d0f-8b0e-e4f057b55239,None
3,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Immunization,55d4e20e-10c7-1d0f-ed71-fdd83dcdeee1,2019-02-22T16:33:40-08:00,"Td (adult), 5 Lf tetanus toxoid, preservative ...",None,urn:uuid:55d4e20e-10c7-1d0f-8b0e-e4f057b55239,None
4,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Immunization,55d4e20e-10c7-1d0f-8d8a-8c5403a6433d,2020-02-28T16:33:40-08:00,"Influenza, split virus, trivalent, PF",None,urn:uuid:55d4e20e-10c7-1d0f-9402-b2e58d043879,None


In [19]:
#combines all the separate row lists into one single table.

timeline_df = pd.DataFrame(
    encounter_rows
    + condition_rows
    + observation_rows
    + medication_rows
    + procedure_rows
    + immunization_rows
)

timeline_df.head()

,patient_id,patient_name,resource_type,resource_id,date,label,code,encounter_ref,subject_ref
0,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Encounter,55d4e20e-10c7-1d0f-7533-e42d4eae3c5d,1965-12-17T16:33:40-08:00,General examination of patient (procedure),NaN,Encounter/55d4e20e-10c7-1d0f-7533-e42d4eae3c5d,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
1,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Encounter,55d4e20e-10c7-1d0f-d25d-1f073d4ed1eb,1969-04-11T16:33:40-08:00,Prenatal visit (regime/therapy),NaN,Encounter/55d4e20e-10c7-1d0f-d25d-1f073d4ed1eb,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
2,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Encounter,55d4e20e-10c7-1d0f-cb8d-9b74f706fbe4,1987-10-30T16:33:40-08:00,General examination of patient (procedure),NaN,Encounter/55d4e20e-10c7-1d0f-cb8d-9b74f706fbe4,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
3,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Encounter,55d4e20e-10c7-1d0f-d9dc-2aae189ec7f2,2003-10-20T02:33:40-07:00,Encounter for symptom (procedure),NaN,Encounter/55d4e20e-10c7-1d0f-d9dc-2aae189ec7f2,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd
4,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Encounter,55d4e20e-10c7-1d0f-f8aa-93427b44527b,2013-02-08T16:33:40-08:00,Encounter for problem (procedure),NaN,Encounter/55d4e20e-10c7-1d0f-f8aa-93427b44527b,urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd


In [20]:
#parse the dates and sort the timeline.
timeline_df["date_parsed"] = pd.to_datetime(timeline_df["date"], errors="coerce", utc=True)
timeline_df = timeline_df.sort_values(["date_parsed", "resource_type", "label"]).reset_index(drop=True)

timeline_df[["date", "date_parsed", "resource_type", "label", "code", "encounter_ref"]].head(30)

,date,date_parsed,resource_type,label,code,encounter_ref
0,1965-12-17T16:33:40-08:00,1965-12-18 00:33:40+00:00,Encounter,General examination of patient (procedure),NaN,Encounter/55d4e20e-10c7-1d0f-7533-e42d4eae3c5d
1,1965-12-17T17:04:41-08:00,1965-12-18 01:04:41+00:00,Condition,Received higher education (finding),224299000,urn:uuid:55d4e20e-10c7-1d0f-7533-e42d4eae3c5d
2,1969-04-11T16:33:40-08:00,1969-04-12 00:33:40+00:00,Encounter,Prenatal visit (regime/therapy),NaN,Encounter/55d4e20e-10c7-1d0f-d25d-1f073d4ed1eb
3,1969-04-23T20:58:04-08:00,1969-04-24 04:58:04+00:00,Condition,Recurrent urinary tract infection (disorder),197927001,urn:uuid:55d4e20e-10c7-1d0f-d25d-1f073d4ed1eb
4,1987-10-30T16:33:40-08:00,1987-10-31 00:33:40+00:00,Encounter,General examination of patient (procedure),NaN,Encounter/55d4e20e-10c7-1d0f-cb8d-9b74f706fbe4
5,1987-10-30T17:16:05-08:00,1987-10-31 01:16:05+00:00,Condition,Refugee (person),446654005,urn:uuid:55d4e20e-10c7-1d0f-cb8d-9b74f706fbe4
6,2003-10-20T02:33:40-07:00,2003-10-20 09:33:40+00:00,Condition,Chronic sinusitis (disorder),40055000,urn:uuid:55d4e20e-10c7-1d0f-d9dc-2aae189ec7f2
7,2003-10-20T02:33:40-07:00,2003-10-20 09:33:40+00:00,Encounter,Encounter for symptom (procedure),NaN,Encounter/55d4e20e-10c7-1d0f-d9dc-2aae189ec7f2
8,2013-02-08T16:33:40-08:00,2013-02-09 00:33:40+00:00,Condition,Loss of teeth (disorder),37320007,urn:uuid:55d4e20e-10c7-1d0f-f8aa-93427b44527b
9,2013-02-08T16:33:40-08:00,2013-02-09 00:33:40+00:00,Encounter,Encounter for problem (procedure),NaN,Encounter/55d4e20e-10c7-1d0f-f8aa-93427b44527b


In [21]:
#total row x column in the final flat event talbe for the one patient
timeline_df.shape  

(662, 10)

timeline_df["resource_type"].value_counts()

In [22]:
len(encounter_rows), len(condition_rows), len(observation_rows), len(medication_rows), len(procedure_rows), len(immunization_rows)

(46, 42, 402, 27, 133, 12)

### the above concluded with one FHIR patient bundle - meaning one flat longitudial event table
## next: timeline inspection + cleanup turning to a good example for all patients.

In [23]:
timeline_df[["date", "resource_type", "label", "encounter_ref"]].head(30)

,date,resource_type,label,encounter_ref
0,1965-12-17T16:33:40-08:00,Encounter,General examination of patient (procedure),Encounter/55d4e20e-10c7-1d0f-7533-e42d4eae3c5d
1,1965-12-17T17:04:41-08:00,Condition,Received higher education (finding),urn:uuid:55d4e20e-10c7-1d0f-7533-e42d4eae3c5d
2,1969-04-11T16:33:40-08:00,Encounter,Prenatal visit (regime/therapy),Encounter/55d4e20e-10c7-1d0f-d25d-1f073d4ed1eb
3,1969-04-23T20:58:04-08:00,Condition,Recurrent urinary tract infection (disorder),urn:uuid:55d4e20e-10c7-1d0f-d25d-1f073d4ed1eb
4,1987-10-30T16:33:40-08:00,Encounter,General examination of patient (procedure),Encounter/55d4e20e-10c7-1d0f-cb8d-9b74f706fbe4
5,1987-10-30T17:16:05-08:00,Condition,Refugee (person),urn:uuid:55d4e20e-10c7-1d0f-cb8d-9b74f706fbe4
6,2003-10-20T02:33:40-07:00,Condition,Chronic sinusitis (disorder),urn:uuid:55d4e20e-10c7-1d0f-d9dc-2aae189ec7f2
7,2003-10-20T02:33:40-07:00,Encounter,Encounter for symptom (procedure),Encounter/55d4e20e-10c7-1d0f-d9dc-2aae189ec7f2
8,2013-02-08T16:33:40-08:00,Condition,Loss of teeth (disorder),urn:uuid:55d4e20e-10c7-1d0f-f8aa-93427b44527b
9,2013-02-08T16:33:40-08:00,Encounter,Encounter for problem (procedure),Encounter/55d4e20e-10c7-1d0f-f8aa-93427b44527b


In [24]:
timeline_df.query("resource_type == 'Observation'")["label"].value_counts().head(30)

label
Cholesterol [Mass/volume] in Serum or Plasma                                                                     14
Cholesterol in HDL [Mass/volume] in Serum or Plasma                                                              14
Cholesterol in LDL [Mass/volume] in Serum or Plasma by Direct assay                                              14
Triglyceride [Mass/volume] in Serum or Plasma                                                                    14
Alanine aminotransferase [Enzymatic activity/volume] in Serum or Plasma                                          11
Albumin [Mass/volume] in Serum or Plasma                                                                         11
Alkaline phosphatase [Enzymatic activity/volume] in Serum or Plasma                                              11
Aspartate aminotransferase [Enzymatic activity/volume] in Serum or Plasma                                        11
Bilirubin.total [Mass/volume] in Serum or Plasma                  

In [25]:
timeline_df.query("resource_type == 'Procedure'")["label"].value_counts().head(30)

label
Depression screening (procedure)                                                                   16
Hospice care (regime/therapy)                                                                      12
Assessment of health and social care needs (procedure)                                             10
Assessment using Morse Fall Scale (procedure)                                                       8
Assessment of substance use (procedure)                                                             6
Medication reconciliation (procedure)                                                               5
Screening for domestic abuse (procedure)                                                            5
Dental consultation and report (procedure)                                                          5
Patient referral for dental care (procedure)                                                        4
Dental care (regime/therapy)                                                

In [26]:
#count missing labels
timeline_df["label"].isna().sum(), timeline_df["date"].isna().sum()

(np.int64(5), np.int64(0))

In [27]:
encounters[0]

{'resourceType': 'Encounter',
 'id': '55d4e20e-10c7-1d0f-7533-e42d4eae3c5d',
 'meta': {'profile': ['http://hl7.org/fhir/us/core/StructureDefinition/us-core-encounter']},
 'identifier': [{'use': 'official',
   'system': 'https://github.com/synthetichealth/synthea',
   'value': '55d4e20e-10c7-1d0f-7533-e42d4eae3c5d'}],
 'status': 'finished',
 'class': {'system': 'http://terminology.hl7.org/CodeSystem/v3-ActCode',
  'code': 'AMB'},
 'type': [{'coding': [{'system': 'http://snomed.info/sct',
     'code': '162673000',
     'display': 'General examination of patient (procedure)'}],
   'text': 'General examination of patient (procedure)'}],
 'subject': {'reference': 'urn:uuid:55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd',
  'display': 'Mrs. Ardelle563 Lauri399 Aufderhar910'},
 'participant': [{'type': [{'coding': [{'system': 'http://terminology.hl7.org/CodeSystem/v3-ParticipationType',
       'code': 'PPRF',
       'display': 'primary performer'}],
     'text': 'primary performer'}],
   'period': {'st

In [28]:
r = encounters[0]

print("type =", r.get("type"))
print("class =", r.get("class"))
print("serviceType =", r.get("serviceType"))
print("status =", r.get("status"))

type = [{'coding': [{'system': 'http://snomed.info/sct', 'code': '162673000', 'display': 'General examination of patient (procedure)'}], 'text': 'General examination of patient (procedure)'}]
class = {'system': 'http://terminology.hl7.org/CodeSystem/v3-ActCode', 'code': 'AMB'}
serviceType = None
status = finished


In [29]:
#make a clean copy
timeline_clean_df = timeline_df.copy()

In [30]:
#remove rows with missing labels
timeline_clean_df = timeline_clean_df[timeline_clean_df["label"].notna()].copy()
timeline_clean_df.shape

(657, 10)

In [31]:
#define noisy observation labels
noisy_observation_labels = {
    "Body Height", 
    "Body Weight",
    "Body mass index (BMI) [Ratio]",
    "Heart rate",
    "Respiratory rate",
    "Pain severity - 0-10 verbal numeric rating [Score] - Reported",
    "Tobacco smoking status",
    "Blood pressure panel with all children optional",
    "Protocol for Responding to and Assessing Patients' Assets, Risks, and Experiences [PRAPARE]",
    "Fall risk level [Morse Fall Scale]",
    
}

In [32]:
#filter the above defined noise obser- emoves unlabeled rows + most repetitive low-value observation rows
timeline_clean_df = timeline_clean_df[
    ~(
        (timeline_clean_df["resource_type"] == "Observation") &
        (timeline_clean_df["label"].isin(noisy_observation_labels))
    )
].copy()

timeline_clean_df["resource_type"].value_counts()

resource_type
Observation          304
Procedure            133
Encounter             46
Condition             42
MedicationRequest     22
Immunization          12
Name: count, dtype: int64

In [33]:
#preview the cleaned timeline
timeline_clean_df[["date", "resource_type", "label", "encounter_ref"]].head(40)

,date,resource_type,label,encounter_ref
0,1965-12-17T16:33:40-08:00,Encounter,General examination of patient (procedure),Encounter/55d4e20e-10c7-1d0f-7533-e42d4eae3c5d
1,1965-12-17T17:04:41-08:00,Condition,Received higher education (finding),urn:uuid:55d4e20e-10c7-1d0f-7533-e42d4eae3c5d
2,1969-04-11T16:33:40-08:00,Encounter,Prenatal visit (regime/therapy),Encounter/55d4e20e-10c7-1d0f-d25d-1f073d4ed1eb
3,1969-04-23T20:58:04-08:00,Condition,Recurrent urinary tract infection (disorder),urn:uuid:55d4e20e-10c7-1d0f-d25d-1f073d4ed1eb
4,1987-10-30T16:33:40-08:00,Encounter,General examination of patient (procedure),Encounter/55d4e20e-10c7-1d0f-cb8d-9b74f706fbe4
5,1987-10-30T17:16:05-08:00,Condition,Refugee (person),urn:uuid:55d4e20e-10c7-1d0f-cb8d-9b74f706fbe4
6,2003-10-20T02:33:40-07:00,Condition,Chronic sinusitis (disorder),urn:uuid:55d4e20e-10c7-1d0f-d9dc-2aae189ec7f2
7,2003-10-20T02:33:40-07:00,Encounter,Encounter for symptom (procedure),Encounter/55d4e20e-10c7-1d0f-d9dc-2aae189ec7f2
8,2013-02-08T16:33:40-08:00,Condition,Loss of teeth (disorder),urn:uuid:55d4e20e-10c7-1d0f-f8aa-93427b44527b
9,2013-02-08T16:33:40-08:00,Encounter,Encounter for problem (procedure),Encounter/55d4e20e-10c7-1d0f-f8aa-93427b44527b


## turn the above one-patient logic into a reusable function that can run it on all patient bundles

In [34]:
#a helper to detect patient bundle files- Some JSON files in the folder are not patient bundles
def is_patient_bundle(data):
    if data.get("resourceType") != "Bundle":
        return False
    entries = data.get("entry", [])
    for entry in entries:
        r = entry.get("resource", {})
        if r.get("resourceType") == "Patient":
            return True
    return False

In [35]:
#important step: create one main function to extract timelines from one bundle
def extract_timeline_from_bundle(file_path):
    import json
    import pandas as pd

    with open(file_path) as fh:
        data = json.load(fh)

    if not is_patient_bundle(data):
        return None, None

    entries = data.get("entry", [])
    resources = [e.get("resource", {}) for e in entries]

    patient_resources = [r for r in resources if r.get("resourceType") == "Patient"]
    encounters = [r for r in resources if r.get("resourceType") == "Encounter"]
    conditions = [r for r in resources if r.get("resourceType") == "Condition"]
    observations = [r for r in resources if r.get("resourceType") == "Observation"]
    med_requests = [r for r in resources if r.get("resourceType") == "MedicationRequest"]
    procedures = [r for r in resources if r.get("resourceType") == "Procedure"]
    immunizations = [r for r in resources if r.get("resourceType") == "Immunization"]

    if not patient_resources:
        return None, None

    patient = patient_resources[0]
    patient_id = patient.get("id")
    patient_name = get_patient_name(patient)

    encounter_rows = []
    for r in encounters:
        encounter_rows.append({
            "patient_id": patient_id,
            "patient_name": patient_name,
            "resource_type": "Encounter",
            "resource_id": r.get("id"),
            "date": normalize_date(r.get("period", {}).get("start")),
            "label": get_encounter_label(r),
            "code": get_code_value(r),
            "encounter_ref": f"Encounter/{r.get('id')}" if r.get("id") else None,
            "subject_ref": get_subject_ref(r),
        })

    condition_rows = []
    for r in conditions:
        condition_rows.append({
            "patient_id": patient_id,
            "patient_name": patient_name,
            "resource_type": "Condition",
            "resource_id": r.get("id"),
            "date": normalize_date(r.get("onsetDateTime") or r.get("recordedDate")),
            "label": get_code_text(r),
            "code": get_code_value(r),
            "encounter_ref": get_encounter_ref(r),
            "subject_ref": get_subject_ref(r),
        })

    observation_rows = []
    for r in observations:
        observation_rows.append({
            "patient_id": patient_id,
            "patient_name": patient_name,
            "resource_type": "Observation",
            "resource_id": r.get("id"),
            "date": normalize_date(
                r.get("effectiveDateTime")
                or r.get("issued")
                or r.get("effectivePeriod", {}).get("start")
            ),
            "label": get_code_text(r),
            "code": get_code_value(r),
            "encounter_ref": get_encounter_ref(r),
            "subject_ref": get_subject_ref(r),
        })

    medication_rows = []
    for r in med_requests:
        med_label = None
        med_code = r.get("medicationCodeableConcept", {})
        if "text" in med_code:
            med_label = med_code["text"]
        else:
            coding = med_code.get("coding", [])
            if coding:
                med_label = coding[0].get("display") or coding[0].get("code")

        medication_rows.append({
            "patient_id": patient_id,
            "patient_name": patient_name,
            "resource_type": "MedicationRequest",
            "resource_id": r.get("id"),
            "date": normalize_date(r.get("authoredOn")),
            "label": med_label,
            "code": None,
            "encounter_ref": get_encounter_ref(r),
            "subject_ref": get_subject_ref(r),
        })

    procedure_rows = []
    for r in procedures:
        performed = r.get("performedDateTime")
        if not performed:
            performed = r.get("performedPeriod", {}).get("start")

        procedure_rows.append({
            "patient_id": patient_id,
            "patient_name": patient_name,
            "resource_type": "Procedure",
            "resource_id": r.get("id"),
            "date": normalize_date(performed),
            "label": get_code_text(r),
            "code": get_code_value(r),
            "encounter_ref": get_encounter_ref(r),
            "subject_ref": get_subject_ref(r),
        })

    immunization_rows = []
    for r in immunizations:
        immunization_rows.append({
            "patient_id": patient_id,
            "patient_name": patient_name,
            "resource_type": "Immunization",
            "resource_id": r.get("id"),
            "date": normalize_date(r.get("occurrenceDateTime")),
            "label": get_vaccine_text(r),
            "code": None,
            "encounter_ref": get_encounter_ref(r),
            "subject_ref": get_subject_ref(r),
        })

    timeline_df = pd.DataFrame(
        encounter_rows
        + condition_rows
        + observation_rows
        + medication_rows
        + procedure_rows
        + immunization_rows
    )

    if timeline_df.empty:
        return None, None

    timeline_df["date_parsed"] = pd.to_datetime(timeline_df["date"], errors="coerce", utc=True)
    timeline_df = timeline_df.sort_values(["date_parsed", "resource_type", "label"]).reset_index(drop=True)

    timeline_clean_df = timeline_df[timeline_df["label"].notna()].copy()

    noisy_observation_labels = {
        "Body Height",
        "Body Weight",
        "Body mass index (BMI) [Ratio]",
        "Heart rate",
        "Respiratory rate",
        "Pain severity - 0-10 verbal numeric rating [Score] - Reported",
        "Tobacco smoking status",
        "Blood pressure panel with all children optional",
        "Protocol for Responding to and Assessing Patients' Assets, Risks, and Experiences [PRAPARE]",
        "Fall risk level [Morse Fall Scale]",
    }

    timeline_clean_df = timeline_clean_df[
        ~(
            (timeline_clean_df["resource_type"] == "Observation") &
            (timeline_clean_df["label"].isin(noisy_observation_labels))
        )
    ].copy()

    return timeline_df, timeline_clean_df

In [36]:
#test the main fucntion- extract_timeline_from_bundle(file_path)
test_timeline_df, test_timeline_clean_df = extract_timeline_from_bundle(example_file)

test_timeline_df.shape, test_timeline_clean_df.shape

((662, 10), (559, 10))

In [37]:
test_timeline_clean_df["resource_type"].value_counts()

resource_type
Observation          304
Procedure            133
Encounter             46
Condition             42
MedicationRequest     22
Immunization          12
Name: count, dtype: int64

In [38]:
#Run it across all patient bundles
all_full = []
all_clean = []
processed_files = []

for f in files:
    full_df, clean_df = extract_timeline_from_bundle(f)
    if full_df is not None:
        full_df["source_file"] = f.name
        clean_df["source_file"] = f.name
        all_full.append(full_df)
        all_clean.append(clean_df)
        processed_files.append(f.name)

len(processed_files), processed_files[:5]

(22,
 ['Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd.json',
  'Bobby524_Nikolaus26_e94ea1e4-7725-20f8-c494-32a002550372.json',
  'Carry843_Jerrie417_Bosco882_8f919ef5-4994-e505-786f-fd2d5a4d895f.json',
  'Cornell131_Ratke343_57ee3fbe-8320-2480-7b90-0e0980bd584c.json',
  'Garfield38_Ferry570_83a1f980-5e41-7be5-445a-882e773a70dc.json'])

In [39]:
#Combine all patients
all_timeline_df = pd.concat(all_full, ignore_index=True)
all_timeline_clean_df = pd.concat(all_clean, ignore_index=True)

all_timeline_df.shape, all_timeline_clean_df.shape

((21684, 11), (17446, 11))

In [43]:
all_timeline_clean_df["resource_type"].value_counts()

resource_type
Observation          9641
Procedure            4113
Encounter            1961
Condition             796
MedicationRequest     631
Immunization          304
Name: count, dtype: int64

In [44]:
all_timeline_clean_df[["patient_name", "date", "resource_type", "label", "source_file"]].head(30)

,patient_name,date,resource_type,label,source_file
0,Ardelle563 Lauri399 Aufderhar910,1965-12-17T16:33:40-08:00,Encounter,General examination of patient (procedure),Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7...
1,Ardelle563 Lauri399 Aufderhar910,1965-12-17T17:04:41-08:00,Condition,Received higher education (finding),Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7...
2,Ardelle563 Lauri399 Aufderhar910,1969-04-11T16:33:40-08:00,Encounter,Prenatal visit (regime/therapy),Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7...
3,Ardelle563 Lauri399 Aufderhar910,1969-04-23T20:58:04-08:00,Condition,Recurrent urinary tract infection (disorder),Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7...
4,Ardelle563 Lauri399 Aufderhar910,1987-10-30T16:33:40-08:00,Encounter,General examination of patient (procedure),Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7...
5,Ardelle563 Lauri399 Aufderhar910,1987-10-30T17:16:05-08:00,Condition,Refugee (person),Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7...
6,Ardelle563 Lauri399 Aufderhar910,2003-10-20T02:33:40-07:00,Condition,Chronic sinusitis (disorder),Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7...
7,Ardelle563 Lauri399 Aufderhar910,2003-10-20T02:33:40-07:00,Encounter,Encounter for symptom (procedure),Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7...
8,Ardelle563 Lauri399 Aufderhar910,2013-02-08T16:33:40-08:00,Condition,Loss of teeth (disorder),Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7...
9,Ardelle563 Lauri399 Aufderhar910,2013-02-08T16:33:40-08:00,Encounter,Encounter for problem (procedure),Ardelle563_Lauri399_Aufderhar910_55d4e20e-10c7...


## Export the timeline tables, Build a patient summary table, making the output data more interpretable

In [42]:

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

In [45]:
all_timeline_df.to_csv(out_dir / "synthea_all_timeline_full.csv", index=False)
all_timeline_clean_df.to_csv(out_dir / "synthea_all_timeline_clean.csv", index=False)

In [46]:
sorted([p.name for p in out_dir.iterdir()])

['synthea_all_timeline_clean.csv', 'synthea_all_timeline_full.csv']

In [47]:
#Build a patient summary table
patient_summary_df = (
    all_timeline_clean_df
    .groupby(["patient_id", "patient_name", "resource_type"]  )
    .size()
    .unstack(fill_value=0)
    .reset_index()
    )
patient_summary_df.head()

resource_type,patient_id,patient_name,Condition,Encounter,Immunization,MedicationRequest,Observation,Procedure
0,070128dc-a521-9b3e-5b64-97c2dcf8c433,Jennie425 Hanh683 Hegmann834,20,33,11,5,29,106
1,10acd136-48d1-7616-1cf8-ea8a2cc7d3f7,Oscar384 Charise827 Langosh790,24,42,18,8,45,125
2,1a9e5686-1d64-882e-366e-bbadccb1973e,Karine844 Olympia319 Price929,39,64,16,27,64,229
3,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,42,46,12,22,304,133
4,57ee3fbe-8320-2480-7b90-0e0980bd584c,Cornell131 Kareem959 Ratke343,13,17,15,1,37,79


In [50]:
#one row one patient in the table, with counts of each event type
patient_summary_df.shape

(22, 8)

In [51]:
#Add first and last event dates per patient
date_summary = (
    all_timeline_clean_df
    .groupby(["patient_id", "patient_name"])
    .agg(
        first_event=("date_parsed", "min"),
        last_event=("date_parsed", "max"),
        total_events=("resource_type", "size")
    )
    .reset_index()
)

date_summary.head()


,patient_id,patient_name,first_event,last_event,total_events
0,070128dc-a521-9b3e-5b64-97c2dcf8c433,Jennie425 Hanh683 Hegmann834,2005-02-05 03:28:31+00:00,2025-09-09 06:06:51+00:00,204
1,10acd136-48d1-7616-1cf8-ea8a2cc7d3f7,Oscar384 Charise827 Langosh790,2009-03-04 04:36:52+00:00,2025-10-28 06:50:51+00:00,262
2,1a9e5686-1d64-882e-366e-bbadccb1973e,Karine844 Olympia319 Price929,1996-08-28 06:20:56+00:00,2026-03-15 19:20:56+00:00,439
3,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,1965-12-18 00:33:40+00:00,2025-12-15 00:33:40+00:00,559
4,57ee3fbe-8320-2480-7b90-0e0980bd584c,Cornell131 Kareem959 Ratke343,2016-04-19 13:18:31+00:00,2024-08-06 15:26:47+00:00,162


In [52]:
patient_summary_df = patient_summary_df.merge(
    date_summary,
    on=["patient_id", "patient_name"],   #anchors to line up the rows
    how="left"           #always keep left part toprevent silent deletion when ndate info missing           
)

patient_summary_df.head()

,patient_id,patient_name,Condition,Encounter,Immunization,MedicationRequest,Observation,Procedure,first_event,last_event,total_events
0,070128dc-a521-9b3e-5b64-97c2dcf8c433,Jennie425 Hanh683 Hegmann834,20,33,11,5,29,106,2005-02-05 03:28:31+00:00,2025-09-09 06:06:51+00:00,204
1,10acd136-48d1-7616-1cf8-ea8a2cc7d3f7,Oscar384 Charise827 Langosh790,24,42,18,8,45,125,2009-03-04 04:36:52+00:00,2025-10-28 06:50:51+00:00,262
2,1a9e5686-1d64-882e-366e-bbadccb1973e,Karine844 Olympia319 Price929,39,64,16,27,64,229,1996-08-28 06:20:56+00:00,2026-03-15 19:20:56+00:00,439
3,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,42,46,12,22,304,133,1965-12-18 00:33:40+00:00,2025-12-15 00:33:40+00:00,559
4,57ee3fbe-8320-2480-7b90-0e0980bd584c,Cornell131 Kareem959 Ratke343,13,17,15,1,37,79,2016-04-19 13:18:31+00:00,2024-08-06 15:26:47+00:00,162


In [53]:
#Save the patient summary
patient_summary_df.to_csv(out_dir / "synthea_patient_summary.csv", index=False)

In [54]:
sorted([p.name for p in out_dir.iterdir()])

['synthea_all_timeline_clean.csv',
 'synthea_all_timeline_full.csv',
 'synthea_patient_summary.csv']

In [59]:
#Quick quality checks, ..nunique()==Number of Unique values.how many unique patients were processed
all_timeline_clean_df["patient_name"].nunique(), all_timeline_clean_df["patient_id"].nunique()

(22, 22)

In [58]:
#show which patients have the richest timelines
patient_summary_df.sort_values("total_events", ascending=False).head(10)

,patient_id,patient_name,Condition,Encounter,Immunization,MedicationRequest,Observation,Procedure,first_event,last_event,total_events
13,9fae51ed-2e3e-ad78-639e-1aef424983bc,Trinidad33 Ray930 Konopelski743,80,749,13,75,3901,990,1952-04-27 07:33:40+00:00,2023-03-18 00:33:40+00:00,5808
19,daea9f69-7b69-ca9e-2185-1e9615226a0d,Maynard46 Hector463 Haley279,109,418,14,230,3324,755,1952-10-03 05:34:00+00:00,2026-03-18 05:03:00+00:00,4850
20,e94ea1e4-7725-20f8-c494-32a002550372,Bobby524 Lawerence6 Nikolaus26,39,36,15,16,373,163,1976-10-12 06:15:40+00:00,2025-12-09 09:25:57+00:00,642
6,76bf625d-c8b8-cfb6-c04c-c9ab69d541e8,Jere230 Julio255 Johnson679,51,47,14,17,321,188,1977-06-09 14:29:20+00:00,2025-11-15 17:47:27+00:00,638
15,a8694f66-b421-9144-739e-99cbba9cf4ff,Keshia89 Stefani254 Orn563,32,48,13,22,389,110,1949-03-29 13:31:44+00:00,2026-02-27 23:31:44+00:00,614
3,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,42,46,12,22,304,133,1965-12-18 00:33:40+00:00,2025-12-15 00:33:40+00:00,559
2,1a9e5686-1d64-882e-366e-bbadccb1973e,Karine844 Olympia319 Price929,39,64,16,27,64,229,1996-08-28 06:20:56+00:00,2026-03-15 19:20:56+00:00,439
18,cb76afc4-df96-8737-07d1-e8907edd3d18,Maragaret140 Carolynn568 Hodkiewicz467,39,31,13,36,151,130,1981-02-23 04:29:25+00:00,2025-09-08 07:09:22+00:00,400
14,a1712c54-00f0-648e-5af7-49cc95248158,Herbert830 Dallas143 Hills818,21,92,17,30,75,144,2007-07-15 06:25:22+00:00,2026-03-08 22:29:12+00:00,379
7,797c8d36-d5f9-0da1-a58d-c6392946ad4a,Scottie437 Lisette854 Cartwright189,45,48,12,9,74,163,1965-12-18 00:33:40+00:00,2009-09-05 00:33:40+00:00,351


In [60]:
#Use a patient from patient_summary_df
top_patient = patient_summary_df.sort_values("total_events", ascending=False).iloc[0]
top_patient_id = top_patient["patient_id"]
top_patient_name = top_patient["patient_name"]

top_patient_id, top_patient_name

('9fae51ed-2e3e-ad78-639e-1aef424983bc', 'Trinidad33 Ray930 Konopelski743')

In [61]:
#subset the clean timeline for that patient
patient_timeline = all_timeline_clean_df[
    all_timeline_clean_df["patient_id"] == top_patient_id
].copy()

patient_timeline.shape

(5808, 11)

In [62]:
#inspect the first rows
patient_timeline[["date", "resource_type", "label"]].head(20)

,date,resource_type,label
11360,1952-04-26T23:33:40-08:00,Condition,Chronic sinusitis (disorder)
11361,1952-04-26T23:33:40-08:00,Encounter,Encounter for symptom (procedure)
11362,1965-12-17T16:33:40-08:00,Encounter,General examination of patient (procedure)
11363,1965-12-17T17:15:31-08:00,Condition,Housing unsatisfactory (finding)
11364,1965-12-17T17:15:31-08:00,Condition,Received higher education (finding)
11365,1965-12-17T17:15:31-08:00,Condition,Refugee (person)
11366,1976-01-02T16:33:40-08:00,Condition,Body mass index 30+ - obesity (finding)
11367,1976-01-02T16:33:40-08:00,Encounter,General examination of patient (procedure)
11368,1976-01-30T10:33:40-08:00,Encounter,Encounter for problem (procedure)
11369,1976-01-30T10:33:40-08:00,MedicationRequest,Hydrocortisone 10 MG/ML Topical Cream


In [63]:
#helper function to get top labels
def top_labels(df, resource_type, n=5):
    subset = df[df["resource_type"] == resource_type]
    if subset.empty:
        return []
    return subset["label"].value_counts().head(n).index.tolist()

In [64]:
#build a deterministic patient summary
def build_patient_narrative(patient_df):
    if patient_df.empty:
        return "No patient events available."

    patient_name = patient_df["patient_name"].dropna().iloc[0]
    patient_id = patient_df["patient_id"].dropna().iloc[0]

    first_event = patient_df["date_parsed"].min()
    last_event = patient_df["date_parsed"].max()

    counts = patient_df["resource_type"].value_counts().to_dict()

    n_encounters = counts.get("Encounter", 0)
    n_conditions = counts.get("Condition", 0)
    n_meds = counts.get("MedicationRequest", 0)
    n_obs = counts.get("Observation", 0)
    n_proc = counts.get("Procedure", 0)
    n_imm = counts.get("Immunization", 0)

    top_conditions = top_labels(patient_df, "Condition", n=5)
    top_meds = top_labels(patient_df, "MedicationRequest", n=5)
    top_procs = top_labels(patient_df, "Procedure", n=5)
    top_imms = top_labels(patient_df, "Immunization", n=5)

    summary = []
    summary.append(
        f"{patient_name} (patient_id: {patient_id}) has a longitudinal clinical timeline "
        f"from {first_event.date()} to {last_event.date()}."
    )
    summary.append(
        f"The cleaned record contains {n_encounters} encounters, {n_conditions} conditions, "
        f"{n_meds} medication requests, {n_proc} procedures, {n_obs} observations, and {n_imm} immunizations."
    )

    if top_conditions:
        summary.append("Most frequent documented conditions: " + "; ".join(top_conditions) + ".")
    if top_meds:
        summary.append("Common medications include: " + "; ".join(top_meds) + ".")
    if top_procs:
        summary.append("Common procedures include: " + "; ".join(top_procs) + ".")
    if top_imms:
        summary.append("Immunizations include: " + "; ".join(top_imms) + ".")

    return "\n\n".join(summary)

In [66]:
#generate the narrative
patient_summary_text = build_patient_narrative(patient_timeline)
print(patient_summary_text)

Trinidad33 Ray930 Konopelski743 (patient_id: 9fae51ed-2e3e-ad78-639e-1aef424983bc) has a longitudinal clinical timeline from 1952-04-27 to 2023-03-18.

The cleaned record contains 749 encounters, 80 conditions, 75 medication requests, 990 procedures, 3901 observations, and 13 immunizations.

Most frequent documented conditions: Medication review due (situation); Gingivitis (disorder); Stress (finding); Full-time employment (finding); Social isolation (finding).

Common medications include: insulin isophane, human 70 UNT/ML / insulin, regular, human 30 UNT/ML Injectable Suspension [Humulin]; losartan potassium 25 MG Oral Tablet; 24 HR tacrolimus 5 MG Extended Release Oral Capsule [Astagraf]; Hydrocortisone 10 MG/ML Topical Cream; Chlorpheniramine Maleate 4 MG Oral Tablet.

Common procedures include: Renal dialysis (procedure); Hospice care (regime/therapy); Depression screening (procedure); Assessment of health and social care needs (procedure); Assessment using Morse Fall Scale (proced

In [69]:
#Greusable summary builder,Use existing one-patient function, but slightly extend it.
def top_labels(df, resource_type, n=5):
    subset = df[df["resource_type"] == resource_type]
    if subset.empty:
        return []
    return subset["label"].value_counts().head(n).index.tolist()


def build_patient_narrative(patient_df):
    if patient_df.empty:
        return "No patient events available."

    patient_name = patient_df["patient_name"].dropna().iloc[0]
    patient_id = patient_df["patient_id"].dropna().iloc[0]

    first_event = patient_df["date_parsed"].min()
    last_event = patient_df["date_parsed"].max()

    counts = patient_df["resource_type"].value_counts().to_dict()

    n_encounters = counts.get("Encounter", 0)
    n_conditions = counts.get("Condition", 0)
    n_meds = counts.get("MedicationRequest", 0)
    n_obs = counts.get("Observation", 0)
    n_proc = counts.get("Procedure", 0)
    n_imm = counts.get("Immunization", 0)

    top_conditions = top_labels(patient_df, "Condition", n=5)
    top_meds = top_labels(patient_df, "MedicationRequest", n=5)
    top_procs = top_labels(patient_df, "Procedure", n=5)
    top_imms = top_labels(patient_df, "Immunization", n=5)

    summary = []
    summary.append(
        f"{patient_name} (patient_id: {patient_id}) has a longitudinal clinical timeline "
        f"from {first_event.date()} to {last_event.date()}."
    )
    summary.append(
        f"The cleaned record contains {n_encounters} encounters, {n_conditions} conditions, "
        f"{n_meds} medication requests, {n_proc} procedures, {n_obs} observations, and {n_imm} immunizations."
    )

    if top_conditions:
        summary.append("Most frequent documented conditions: " + "; ".join(top_conditions) + ".")
    if top_meds:
        summary.append("Common medications include: " + "; ".join(top_meds) + ".")
    if top_procs:
        summary.append("Common procedures include: " + "; ".join(top_procs) + ".")
    if top_imms:
        summary.append("Immunizations include: " + "; ".join(top_imms) + ".")

    return "\n\n".join(summary)

In [70]:
#generate summaries for all patients
patient_summaries = []

for patient_id, patient_df in all_timeline_clean_df.groupby("patient_id"):
    patient_name = patient_df["patient_name"].dropna().iloc[0]
    summary_text = build_patient_narrative(patient_df)

    patient_summaries.append({
        "patient_id": patient_id,
        "patient_name": patient_name,
        "summary_text": summary_text,
        "total_events": len(patient_df),
        "first_event": patient_df["date_parsed"].min(),
        "last_event": patient_df["date_parsed"].max(),
    })

patient_summaries_df = pd.DataFrame(patient_summaries)
patient_summaries_df.head()

,patient_id,patient_name,summary_text,total_events,first_event,last_event
0,070128dc-a521-9b3e-5b64-97c2dcf8c433,Jennie425 Hanh683 Hegmann834,Jennie425 Hanh683 Hegmann834 (patient_id: 0701...,204,2005-02-05 03:28:31+00:00,2025-09-09 06:06:51+00:00
1,10acd136-48d1-7616-1cf8-ea8a2cc7d3f7,Oscar384 Charise827 Langosh790,Oscar384 Charise827 Langosh790 (patient_id: 10...,262,2009-03-04 04:36:52+00:00,2025-10-28 06:50:51+00:00
2,1a9e5686-1d64-882e-366e-bbadccb1973e,Karine844 Olympia319 Price929,Karine844 Olympia319 Price929 (patient_id: 1a9...,439,1996-08-28 06:20:56+00:00,2026-03-15 19:20:56+00:00
3,55d4e20e-10c7-1d0f-ac7d-2a6a2aea29fd,Ardelle563 Lauri399 Aufderhar910,Ardelle563 Lauri399 Aufderhar910 (patient_id: ...,559,1965-12-18 00:33:40+00:00,2025-12-15 00:33:40+00:00
4,57ee3fbe-8320-2480-7b90-0e0980bd584c,Cornell131 Kareem959 Ratke343,Cornell131 Kareem959 Ratke343 (patient_id: 57e...,162,2016-04-19 13:18:31+00:00,2024-08-06 15:26:47+00:00


In [71]:
#save batch summaries
patient_summaries_df.to_csv(out_dir / "synthea_patient_narratives.csv", index=False)
patient_summaries_df.shape

(22, 6)

## continue to 02_fhir_timeline_summary_agent.ipynb